# 01 — Exploratory Data Analysis (EDA)

Phân tích khám phá dữ liệu mực nước Quảng Hà.

**Mục tiêu:**
1. Hiểu cấu trúc dữ liệu (cột, kiểu, missing values)
2. Trực quan hóa phân bố và xu hướng
3. Phát hiện chu kỳ triều và mùa
4. Kiểm tra tương quan giữa các biến

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Tải dữ liệu

In [ ]:
# Doc du lieu tu file CSV
# Neu chua co du lieu that, chay: python scripts/generate_data.py
from src.data.loader import load_raw_data

df = load_raw_data()
print(f'Kich thuoc: {df.shape}')
print(f'Thoi gian: {df.index[0]} → {df.index[-1]}')
df.head()

In [ ]:
# Thong ke mo ta
df.describe()

In [ ]:
# Kiem tra missing values
print('Missing values:')
print(df.isnull().sum())

## 2. Trực quan hóa chuỗi thời gian

In [ ]:
# Ve chuoi thoi gian toan bo
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for ax, col, label in zip(axes, df.columns, 
                          ['Luu luong Ha Coi (m3/s)', 'Luu luong Tai Chi (m3/s)',
                           'Muc nuoc bien (m)', 'Muc nuoc Quang Ha (m)']):
    ax.plot(df.index, df[col], linewidth=0.3)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)

axes[0].set_title('Du lieu thuy van Quang Ha (MIKE11 output)')
plt.xlabel('Thoi gian')
plt.tight_layout()
plt.show()

## 3. Phân bố mực nước

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
axes[0].hist(df['water_level'], bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(1.06, color='orange', linestyle='--', label='P95 = 1.06 m (Warning)')
axes[0].axvline(1.24, color='red', linestyle='--', label='P99 = 1.24 m (Exceedance)')
axes[0].set_xlabel('Muc nuoc (m)')
axes[0].set_ylabel('Tan suat')
axes[0].set_title('Phan bo muc nuoc Quang Ha')
axes[0].legend()

# Boxplot theo nam
df_temp = df.copy()
df_temp['year'] = df_temp.index.year
df_temp.boxplot(column='water_level', by='year', ax=axes[1], grid=False)
axes[1].set_xlabel('Nam')
axes[1].set_ylabel('Muc nuoc (m)')
axes[1].set_title('Muc nuoc theo nam')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Tương quan

In [ ]:
# Heatmap tuong quan
corr = df.corr(method='spearman')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Spearman Correlation')
plt.tight_layout()
plt.show()

## 5. Tự tương quan (Autocorrelation)

In [ ]:
# Tinh tuong quan tu (autocorrelation) cua muc nuoc
max_lag = 60
autocorrs = [df['water_level'].autocorr(lag=lag) for lag in range(1, max_lag + 1)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, max_lag + 1), autocorrs, width=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(14.76, color='red', linestyle='--', label='Chu ky trieu ~14.76 ngay')
ax.set_xlabel('Lag (ngay)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Tu tuong quan cua muc nuoc Quang Ha')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Mùa và chu kỳ

In [ ]:
# Muc nuoc trung binh theo thang
monthly = df.groupby(df.index.month)['water_level'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(monthly.index, monthly['mean'], yerr=monthly['std'], 
            marker='o', capsize=3, linewidth=2)
ax.set_xlabel('Thang')
ax.set_ylabel('Muc nuoc TB (m)')
ax.set_title('Muc nuoc trung binh theo thang (co do lech chuan)')
ax.set_xticks(range(1, 13))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Nhan xet: Muc nuoc cao hon vao mua he/thu (thang 6-10) do mua lu.')

## 7. Temporal Split

In [ ]:
from src.data.loader import create_temporal_split

splits = create_temporal_split(df)
print(f'Train: {splits["train"].shape[0]} ngay ({splits["train"].index[0].date()} → {splits["train"].index[-1].date()})')
print(f'Val:   {splits["val"].shape[0]} ngay ({splits["val"].index[0].date()} → {splits["val"].index[-1].date()})')
print(f'Test:  {splits["test"].shape[0]} ngay ({splits["test"].index[0].date()} → {splits["test"].index[-1].date()})')

## Tóm tắt EDA

**Nhận xét chính:**
1. Dữ liệu có chu kỳ triều rõ ràng (~14.76 ngày)
2. Mực nước cao hơn vào mùa hè/thu (tháng 6-10) do mưa lũ
3. Phân bố mực nước lệch phải, có đuôi dài (extreme events)
4. Tương quan cao giữa mực nước và mực nước biển